In [3]:
import os
os.chdir(r'D:\8th semester\Machine Learning Lab')

In [4]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [5]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [6]:
def CNN():
    input_data = Input(shape=(time_steps, num_features))
    x1 = Conv1D(16, 2, activation="relu")(input_data)
    x2 = Conv1D(16, 2, activation="relu")(x1)
    flatten = Flatten()(x2)
    output_data = Dense(1)(flatten)
    model = Model(input_data, output_data)
    return model

In [7]:
model1 = CNN()
model1.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 conv1d (Conv1D)             (None, 23, 16)            688       
                                                                 
 conv1d_1 (Conv1D)           (None, 22, 16)            528       
                                                                 
 flatten (Flatten)           (None, 352)               0         
                                                                 
 dense (Dense)               (None, 1)                 353       
                                                                 
Total params: 1569 (6.13 KB)
Trainable params: 1569 (6.13 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [8]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [10]:
checkpoints = r'D:\8th semester\Machine Learning Lab\par_data_set'
OUTPUT_PATH = r'D:\8th semester\Machine Learning Lab\par_data_set'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [11]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [12]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =CNN()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [13]:
import os
path_dataset =r'D:\8th semester\Machine Learning Lab\par_data_set'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [14]:
time_steps=24
num_features=21

In [15]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.006036520004272461 sec


In [16]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/10


14/27 [==============>...............] - ETA: 0s - loss: 0.1920 - mae: 0.1920 - mape: 82.7404  
Epoch 1: val_loss improved from inf to 0.08082, saving model to D:\8th semester\Machine Learning Lab\par_data_set
INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


27/27 [==============================] - 2s 51ms/step - loss: 0.1454 - mae: 0.1454 - mape: 65.6087 - val_loss: 0.0808 - val_mae: 0.0808 - val_mape: 28.6351
Epoch 2/10
20/27 [=====================>........] - ETA: 0s - loss: 0.0751 - mae: 0.0751 - mape: 40.3462
Epoch 2: val_loss improved from 0.08082 to 0.06051, saving model to D:\8th semester\Machine Learning Lab\par_data_set
INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


27/27 [==============================] - 1s 27ms/step - loss: 0.0736 - mae: 0.0736 - mape: 39.9675 - val_loss: 0.0605 - val_mae: 0.0605 - val_mape: 19.2811
Epoch 3/10
23/27 [========================>.....] - ETA: 0s - loss: 0.0636 - mae: 0.0636 - mape: 31.3853
Epoch 3: val_loss did not improve from 0.06051
27/27 [==============================] - 0s 7ms/step - loss: 0.0630 - mae: 0.0630 - mape: 33.2341 - val_loss: 0.0626 - val_mae: 0.0626 - val_mape: 19.8681
Epoch 4/10
17/27 [=================>............] - ETA: 0s - loss: 0.0578 - mae: 0.0578 - mape: 30.1975
Epoch 4: val_loss improved from 0.06051 to 0.05660, saving model to D:\8th semester\Machine Learning Lab\par_data_set
INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


...

27/27 [==============================] - 1s 29ms/step - loss: 0.0539 - mae: 0.0539 - mape: 24.0535 - val_loss: 0.0497 - val_mae: 0.0497 - val_mape: 15.5935
Epoch 6/10
16/27 [================>.............] - ETA: 0s - loss: 0.0509 - mae: 0.0509 - mape: 22.0308
Epoch 6: val_loss did not improve from 0.04965
27/27 [==============================] - 0s 9ms/step - loss: 0.0499 - mae: 0.0499 - mape: 21.2060 - val_loss: 0.0571 - val_mae: 0.0571 - val_mape: 19.8182
Epoch 7/10
18/27 [===================>..........] - ETA: 0s - loss: 0.0470 - mae: 0.0470 - mape: 18.3134
Epoch 7: val_loss did not improve from 0.04965
27/27 [==============================] - 0s 12ms/step - loss: 0.0474 - mae: 0.0474 - mape: 19.0234 - val_loss: 0.0512 - val_mae: 0.0512 - val_mape: 17.1642
Epoch 8/10
19/27 [====================>.........] - ETA: 0s - loss: 0.0438 - mae: 0.0438 - mape: 19.2446
Epoch 8: val_loss improved from 0.04965 to 0.04889, saving model to D:\8th semester\Machine Learning Lab\par_data_set
INFO:t

INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


27/27 [==============================] - 1s 27ms/step - loss: 0.0436 - mae: 0.0436 - mape: 19.2303 - val_loss: 0.0489 - val_mae: 0.0489 - val_mape: 16.8393
Epoch 9/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0427 - mae: 0.0427 - mape: 17.9411
Epoch 9: val_loss improved from 0.04889 to 0.04591, saving model to D:\8th semester\Machine Learning Lab\par_data_set
INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


27/27 [==============================] - 1s 30ms/step - loss: 0.0426 - mae: 0.0426 - mape: 18.3545 - val_loss: 0.0459 - val_mae: 0.0459 - val_mape: 15.9086
Epoch 10/10
19/27 [====================>.........] - ETA: 0s - loss: 0.0383 - mae: 0.0383 - mape: 17.1856
Epoch 10: val_loss did not improve from 0.04591
27/27 [==============================] - 0s 9ms/step - loss: 0.0379 - mae: 0.0379 - mape: 16.8530 - val_loss: 0.0507 - val_mae: 0.0507 - val_mape: 18.0596


In [17]:

model = load_model(r'D:\8th semester\Machine Learning Lab\par_data_set')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 77ms/step
Mean Absolute Error (MAE): 1441.94
Median Absolute Error (MedAE): 1239.69
Mean Squared Error (MSE): 2257412.26
Root Mean Squared Error (RMSE): 1502.47
Mean Absolute Percentage Error (MAPE): 9.18 %
Median Absolute Percentage Error (MDAPE): 8.01 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


In [20]:
checkpoints = r'D:\8th semester\Machine Learning Lab\par_data_set'
model=r'D:\8th semester\Machine Learning Lab\par_data_set'
start_epoch= 10

In [21]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading D:\8th semester\Machine Learning Lab\par_data_set...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [22]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0367 - mae: 0.0367 - mape: 15.7673 
Epoch 1: val_loss improved from inf to 0.04866, saving model to D:\8th semester\Machine Learning Lab\par_data_set
INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


27/27 [==============================] - 2s 40ms/step - loss: 0.0367 - mae: 0.0367 - mape: 15.3635 - val_loss: 0.0487 - val_mae: 0.0487 - val_mape: 16.8293
Epoch 2/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0353 - mae: 0.0353 - mape: 14.7272
Epoch 2: val_loss did not improve from 0.04866
27/27 [==============================] - 1s 21ms/step - loss: 0.0357 - mae: 0.0357 - mape: 14.5544 - val_loss: 0.0494 - val_mae: 0.0494 - val_mape: 17.2266
Epoch 3/10
17/27 [=================>............] - ETA: 0s - loss: 0.0364 - mae: 0.0364 - mape: 13.9160
Epoch 3: val_loss improved from 0.04866 to 0.04731, saving model to D:\8th semester\Machine Learning Lab\par_data_set
INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


27/27 [==============================] - 1s 25ms/step - loss: 0.0356 - mae: 0.0356 - mape: 14.5471 - val_loss: 0.0473 - val_mae: 0.0473 - val_mape: 16.4972
Epoch 4/10
17/27 [=================>............] - ETA: 0s - loss: 0.0354 - mae: 0.0354 - mape: 13.7355
Epoch 4: val_loss improved from 0.04731 to 0.04631, saving model to D:\8th semester\Machine Learning Lab\par_data_set
INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


...

27/27 [==============================] - 1s 26ms/step - loss: 0.0351 - mae: 0.0351 - mape: 14.3501 - val_loss: 0.0463 - val_mae: 0.0463 - val_mape: 16.0214
Epoch 5/10
18/27 [===================>..........] - ETA: 0s - loss: 0.0351 - mae: 0.0351 - mape: 14.0798
Epoch 5: val_loss did not improve from 0.04631
27/27 [==============================] - 0s 8ms/step - loss: 0.0348 - mae: 0.0348 - mape: 14.0569 - val_loss: 0.0475 - val_mae: 0.0475 - val_mape: 16.2796
Epoch 6/10
17/27 [=================>............] - ETA: 0s - loss: 0.0357 - mae: 0.0357 - mape: 15.1286
Epoch 6: val_loss improved from 0.04631 to 0.04570, saving model to D:\8th semester\Machine Learning Lab\par_data_set
INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


27/27 [==============================] - 1s 28ms/step - loss: 0.0346 - mae: 0.0346 - mape: 14.5463 - val_loss: 0.0457 - val_mae: 0.0457 - val_mape: 15.6260
Epoch 7/10
17/27 [=================>............] - ETA: 0s - loss: 0.0346 - mae: 0.0346 - mape: 14.8296
Epoch 7: val_loss improved from 0.04570 to 0.04227, saving model to D:\8th semester\Machine Learning Lab\par_data_set
INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


INFO:tensorflow:Assets written to: D:\8th semester\Machine Learning Lab\par_data_set\assets


27/27 [==============================] - 1s 30ms/step - loss: 0.0345 - mae: 0.0345 - mape: 14.1740 - val_loss: 0.0423 - val_mae: 0.0423 - val_mape: 14.4406
Epoch 8/10
14/27 [==============>...............] - ETA: 0s - loss: 0.0337 - mae: 0.0337 - mape: 13.0030
Epoch 8: val_loss did not improve from 0.04227
27/27 [==============================] - 0s 9ms/step - loss: 0.0342 - mae: 0.0342 - mape: 13.9625 - val_loss: 0.0469 - val_mae: 0.0469 - val_mape: 16.0186
Epoch 9/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0333 - mae: 0.0333 - mape: 13.5943
Epoch 9: val_loss did not improve from 0.04227
27/27 [==============================] - 0s 8ms/step - loss: 0.0337 - mae: 0.0337 - mape: 13.8294 - val_loss: 0.0456 - val_mae: 0.0456 - val_mape: 15.5132
Epoch 10/10
24/27 [=========================>....] - ETA: 0s - loss: 0.0335 - mae: 0.0335 - mape: 13.4956
Epoch 10: val_loss did not improve from 0.04227
27/27 [==============================] - 0s 7ms/step - loss: 0.0333 - mae: 0

In [23]:

model = load_model(r'D:\8th semester\Machine Learning Lab\par_data_set')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 55ms/step
Mean Absolute Error (MAE): 1825.51
Median Absolute Error (MedAE): 1679.06
Mean Squared Error (MSE): 3514267.87
Root Mean Squared Error (RMSE): 1874.64
Mean Absolute Percentage Error (MAPE): 11.64 %
Median Absolute Percentage Error (MDAPE): 10.85 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


# lab report 

## Lab 1